In [15]:
from dataclasses import dataclass
from openai import OpenAI
import os

@dataclass(frozen=True)
class Provider:
    """ One provider to reliably route requests across all inference providers """
    name: str
    env_var: str 
    is_free: bool
    base_url: str | None
    model: str

PROVIDERS=[
    Provider("OpenAI", "OPENAI_API_KEY", True, None,"gpt-4o-mini"),
    Provider("Groq", "GROQ_API_KEY", True, "https://api.groq.com/openai/v1","openai/gpt-oss-120b"),   
]

def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider

    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set, add one of {expected} to your environment variables")
def build_client(provider: Provider) -> OpenAI:

    
    api_key= os.getenv(provider.env_var)
    if provider.base_url is None:
        return OpenAI(api_key=api_key)
    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url
    )
def have_any_key() -> bool:
    return any(os.getenv(p.env_var) for p in PROVIDERS)
print("Found a provider key. " if have_any_key() else "No provider key found")


Found a provider key. 


In [16]:
def llm_reply(prompt: str) -> str:
    provider= select_provider()
    print(f" Using {provider.name} provider")
    client=build_client(provider)
    result= client.chat.completions.create(
        model=provider.model,
        max_tokens=200,
        messages=[{
            'role':"user",
            "content":prompt
        }]
    )

    return result.choices[0].message.content

In [17]:
prompt =" Who was the PM of UK 2020 ? answer in a single sentence"

try:
    print(llm_reply(prompt))
except Exception as e:
    print(f" Error: {e}")

 Using Groq provider
The Prime Minister of the United Kingdom in 2020 was Boris Johnson.


In [18]:
def chat_reply(messages: list[dict])-> str:
    provider=select_provider()
    client= build_client(provider)
    result= client.chat.completions.create(
        model=provider.model,
        max_tokens=1000,
        messages=messages
    )
    return result.choices[0].message.content


In [19]:
conversation=[]
conversation.append({"role":"user","content":"What is the capital of France"})
reply_from_llm=chat_reply(conversation)
print(reply_from_llm)

The capital of France is **Paris**.


In [20]:
conversation.append({
    "role":"assistant",
    "content":reply_from_llm
})
conversation

[{'role': 'user', 'content': 'What is the capital of France'},
 {'role': 'assistant', 'content': 'The capital of France is **Paris**.'}]

In [21]:
conversation.append({"role":"user", "content":"What is their GDP ?"})
conversation

[{'role': 'user', 'content': 'What is the capital of France'},
 {'role': 'assistant', 'content': 'The capital of France is **Paris**.'},
 {'role': 'user', 'content': 'What is their GDP ?'}]

In [22]:
llm_reply=chat_reply(conversation)
print(llm_reply)

France’s Gross Domestic Product (GDP) – the total market value of all final goods and services produced within the country in a given year – is among the largest in the world.

| Year (latest available) | Nominal GDP (USD) | Nominal GDP (Euro) | GDP per capita (USD) | Source |
|--------------------------|-------------------|--------------------|----------------------|--------|
| **2023** (est.) | **≈ $3.1 trillion** | **≈ €2.9 trillion** | **≈ $46,000** | International Monetary Fund (IMF) World Economic Outlook, April 2024; Eurostat |
| **2022** (actual) | **≈ $2.94 trillion** | **≈ €2.78 trillion** | **≈ $44,600** | World Bank, World Development Indicators; OECD |

### Key points

1. **Ranking** – France is typically the **7th‑largest economy** in the world by nominal GDP (behind the United States, China, Japan, Germany, India, and the United Kingdom as of 2023).

2. **Growth trend** – After a contraction in 2020 due to the COVID‑19 pandemic, France’s economy rebounded with real GDP g